In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasClassifier
import pickle

In [14]:
df=pd.read_csv("Churn_Modelling.csv")

In [15]:
df.drop(["RowNumber", "CustomerId", "Surname"], axis=1, inplace=True)
label_encoder=LabelEncoder()
df["Gender"]=label_encoder.fit_transform(df["Gender"])
onehotencoder=OneHotEncoder(sparse_output=False)
encoded=onehotencoder.fit_transform(df[["Geography"]])
encoded_df=pd.DataFrame(encoded,columns=onehotencoder.get_feature_names_out(["Geography"]))
df=df.drop("Geography",axis=1)
df=pd.concat([df,encoded_df],axis=1)
x=df.drop("Exited",axis=1)
y=df["Exited"]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [16]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

with open('label_encoder.pkl','wb') as file:
    pickle.dump(label_encoder,file)

with open('onehotencoder.pkl','wb') as file:
    pickle.dump(onehotencoder,file)

In [31]:
## Define a function to create the model and try different parameters
def create_model(neurons=64, layers=1):
    model=Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(x_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='relu'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model

In [32]:
model=KerasClassifier(layers=1,neurons=32,build_fn=create_model,epochs=50,batch_size=10,verbose=0)


In [35]:
param_grid={
    'neurons': [32, 64 , 4],
    'layers': [1, 2, 3],
    'epochs': [10,20]
}

In [36]:
from sklearn.model_selection import GridSearchCV
grid=GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=-1,cv=3)
grid_result=grid.fit(x_train,y_train)

print(f"Best: {grid_result.best_score_} using {grid_result.best_params_}")

e:\alternative of C\anaconda\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
e:\alternative of C\anaconda\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best: 0.8531243234505664 using {'epochs': 20, 'layers': 3, 'neurons': 32}
